In [ ]:
# Day 2 – Prompt Engineering
## Project: Customer Complaint Classifier
Today's concepts:
- System prompts and few-shot prompting
- JSON output
- Python dictionaries and functions
- Decorators and error handling
- Conversation memory

In [9]:
import json
import functools
import os
from getpass import getpass
from langchain_groq import ChatGroq

In [3]:
SYSTEM_PROMPT = """
You are a customer complaint classifier.
Classify complaints into:
Payment, Delivery, Product, Refund, Other.
Priority: Low, Medium, High.
Return:
category, priority, reason, sentiment, recommended_action.
Use only the given information.
Keep the response in valid JSON.
"""

In [4]:
examples = """
Complaint: Money was deducted twice.
Output: {"category":"Payment","priority":"High","reason":"Duplicate charge","sentiment":"Negative","recommended_action":"Check payment"}
Complaint: My package arrived late.
Output: {"category":"Delivery","priority":"Medium","reason":"Late delivery","sentiment":"Negative","recommended_action":"Check delivery status"}
Complaint: The product is damaged.
Output: {"category":"Product","priority":"High","reason":"Damaged product","sentiment":"Negative","recommended_action":"Arrange replacement"}
"""

In [5]:
categories = {
    "Payment": "Money or transaction issues",
    "Delivery": "Shipping or late delivery",
    "Product": "Damaged or defective items",
    "Refund": "Refund-related issues",
    "Other": "General complaints"
}
print(categories)

{'Payment': 'Money or transaction issues', 'Delivery': 'Shipping or late delivery', 'Product': 'Damaged or defective items', 'Refund': 'Refund-related issues', 'Other': 'General complaints'}


In [10]:
def classify_complaint(complaint):
    prompt = f"""
    {SYSTEM_PROMPT}
    Examples:
    {examples}
    New complaint: {complaint}
    Return JSON only.
    """
    response = llm.invoke(prompt)
    return json.loads(response.content)

In [11]:
complaint = "I paid twice but received only one order."
result = classify_complaint(complaint)
print(result)

{'category': 'Payment', 'priority': 'High', 'reason': 'Duplicate charge', 'sentiment': 'Negative', 'recommended_action': 'Check payment'}


In [12]:
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("Running:", func.__name__)
        return func(*args, **kwargs)
    return wrapper

In [13]:
@log_call
def check_complaint(text):
    return classify_complaint(text)

check_complaint("My refund has not arrived.")

Running: check_complaint


{'category': 'Refund',
 'priority': 'High',
 'reason': 'Refund not received',
 'sentiment': 'Negative',
 'recommended_action': 'Check refund status'}

In [14]:
def safe_classify(text):
    try:
        return classify_complaint(text)
    except json.JSONDecodeError:
        return {"error": "Invalid JSON response"}
    except Exception as e:
        return {"error": str(e)}
safe_classify("My order was never delivered.")

{'category': 'Delivery',
 'priority': 'High',
 'reason': 'Missing delivery',
 'sentiment': 'Negative',
 'recommended_action': 'Check delivery status'}

In [15]:
history = []
MAX_TURNS = 4
def chat(text):
    history.append({"role": "user", "content": text})
    recent = history[-MAX_TURNS:]
    reply = llm.invoke(
        [("system", SYSTEM_PROMPT)] +
        [(m["role"], m["content"]) for m in recent]
    ).content
    history.append({"role": "assistant", "content": reply})
    return reply

In [16]:
print(chat("My name is Bhagya."))
print(chat("I ordered a chocolate cake."))
print(chat("What is my name and what did I order?"))

{"category":"Other","priority":"Low","reason":"No complaint content provided","sentiment":"Neutral","recommended_action":"No action required"}
{"category":"Other","priority":"Low","reason":"No complaint or issue mentioned","sentiment":"Neutral","recommended_action":"No action required"}
I’m sorry, but I don’t have access to that information.


In [17]:
MEMORY_PROMPT = """
You are a customer support assistant.
Remember the customer's name and order from the conversation.
Answer questions using the conversation history.
Do not invent missing information.
"""
history = []

def chat(text):
    history.append(("human", text))
    reply = llm.invoke(
        [("system", MEMORY_PROMPT)] + history[-8:]
    ).content
    history.append(("ai", reply))
    return reply

In [18]:
print(chat("My name is Bhagya."))
print(chat("I ordered a chocolate cake."))
print(chat("What is my name and what did I order?"))
tests = [
    "My payment was deducted twice.",
    "My parcel is three days late.",
    "The item I received is broken.",
    "I have not received my refund.",
    "I received the wrong product."
]

for complaint in tests:
    print("\nComplaint:", complaint)
    print(safe_classify(complaint))

Hello Bhagya! How can I assist you today? If you have an order number or a specific question, feel free to share it.
Got it! You’ve ordered a chocolate cake, Bhagya. How can I help you with that order?
Your name is Bhagya, and you ordered a chocolate cake.

Complaint: My payment was deducted twice.
{'category': 'Payment', 'priority': 'High', 'reason': 'Duplicate charge', 'sentiment': 'Negative', 'recommended_action': 'Check payment'}

Complaint: My parcel is three days late.
{'category': 'Delivery', 'priority': 'Medium', 'reason': 'Late delivery', 'sentiment': 'Negative', 'recommended_action': 'Check delivery status'}

Complaint: The item I received is broken.
{'category': 'Product', 'priority': 'High', 'reason': 'Damaged product', 'sentiment': 'Negative', 'recommended_action': 'Arrange replacement'}

Complaint: I have not received my refund.
{'category': 'Refund', 'priority': 'High', 'reason': 'Refund not received', 'sentiment': 'Negative', 'recommended_action': 'Check refund status'}

In [19]:
tests = [
    "My payment was deducted twice for one order.",
    "My delivery is three days late.",
    "The product I received is damaged.",
    "I requested a refund but haven't received it.",
    "I received the wrong product.",
    "The website charged me but my order was not placed.",
    "My package tracking has not updated for two days.",
    "The product stopped working after one day.",
    "I want to cancel my order and get my money back.",
    "I am unable to log in to my account."
]
for i, complaint in enumerate(tests, 1):
    print(f"\nTest {i}: {complaint}")
    print(safe_classify(complaint))


Test 1: My payment was deducted twice for one order.
{'category': 'Payment', 'priority': 'High', 'reason': 'Duplicate charge', 'sentiment': 'Negative', 'recommended_action': 'Check payment'}

Test 2: My delivery is three days late.
{'category': 'Delivery', 'priority': 'Medium', 'reason': 'Late delivery', 'sentiment': 'Negative', 'recommended_action': 'Check delivery status'}

Test 3: The product I received is damaged.
{'category': 'Product', 'priority': 'High', 'reason': 'Damaged product', 'sentiment': 'Negative', 'recommended_action': 'Arrange replacement'}

Test 4: I requested a refund but haven't received it.
{'category': 'Refund', 'priority': 'High', 'reason': 'Refund not received', 'sentiment': 'Negative', 'recommended_action': 'Check refund status'}

Test 5: I received the wrong product.
{'category': 'Product', 'priority': 'High', 'reason': 'Wrong product received', 'sentiment': 'Negative', 'recommended_action': 'Send correct product'}

Test 6: The website charged me but my orde